# LinUCB Implementation from Scratch
## Build the industry-standard contextual bandit algorithm

**Goal:** Implement LinUCB (Linear Upper Confidence Bound) and see it learn the relationship between context features and arm rewards.

**Key Insight:** LinUCB combines ridge regression (for prediction) with confidence bounds (for exploration). It's UCB1 extended to contextual settings with principled uncertainty quantification.

**Time:** 15 minutes

In [ ]:
callout("** LinUCB combines ridge regression (for prediction) with confidence bounds (for exploration). It's UCB1 extended to contextual settings with principl", kind="insight")

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
learning_objectives(["**LinUCB combines prediction and exploration** — Ridge regression for expected reward + confidence bounds for uncertainty", "**It recovers the linear structure** — Learned weights closely match true weights after sufficient data", "**Alpha controls exploration** — Too small = exploitation-heavy, too large = exploration-heavy; α=1.0 is a good default", "**Regret is sublinear** — Cumulative regret grows slower over time as the model improves", "**Accuracy improves to 80-90%+** — After learning, LinUCB chooses the optimal arm in most contexts"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Cell 2: Implement LinUCB Class

The algorithm maintains ridge regression for each arm and uses prediction uncertainty for exploration.

In [ ]:
class LinUCB:
    """Linear Upper Confidence Bound for contextual bandits."""
    
    def __init__(self, n_arms, context_dim, alpha=1.0, lambda_=1.0):
        self.n_arms = n_arms
        self.d = context_dim
        self.alpha = alpha  # Exploration parameter
        # Initialize ridge regression for each arm
        self.A = [lambda_ * np.eye(context_dim) for _ in range(n_arms)]
        self.b = [np.zeros(context_dim) for _ in range(n_arms)]
    
    def choose_arm(self, context):
        """Choose arm with highest UCB score."""
        ucb_scores = []
        for a in range(self.n_arms):
            # Solve for theta (ridge regression weights)
            theta = np.linalg.solve(self.A[a], self.b[a])
            # Predicted reward
            pred = context @ theta
            # Uncertainty (standard error)
            std = np.sqrt(context @ np.linalg.solve(self.A[a], context))
            # UCB = prediction + exploration bonus
            ucb = pred + self.alpha * std
            ucb_scores.append(ucb)
        return np.argmax(ucb_scores)
    
    def update(self, arm, context, reward):
        """Update ridge regression for chosen arm."""
        self.A[arm] += np.outer(context, context)
        self.b[arm] += reward * context
    
    def get_weights(self, arm):
        """Get learned weights for an arm."""
        return np.linalg.solve(self.A[arm], self.b[arm])

# Quick test
bandit = LinUCB(n_arms=3, context_dim=2, alpha=1.0)
context = np.array([0.5, -0.3])
arm = bandit.choose_arm(context)
print(f"Chosen arm: {arm}")
print(f"LinUCB initialized successfully!")

## Cell 3: Create Synthetic Commodity Environment

3 arms (Energy, Metals, Agriculture) with rewards that depend linearly on context features.

In [ ]:
class LinearCommodityEnvironment:
    """Commodity environment with linear reward structure."""
    
    def __init__(self):
        self.arms = ['Energy', 'Metals', 'Agriculture']
        # True linear weights for each arm
        # Features: [volatility, term_structure, seasonality]
        self.true_theta = np.array([
            [0.10, 0.15, 0.05],   # Energy: likes low vol, contango, neutral season
            [-0.08, -0.05, 0.03], # Metals: likes high vol, backwardation
            [0.02, 0.08, 0.12]    # Agriculture: likes seasonal factors
        ])
        self.noise_std = 0.05
    
    def get_context(self):
        """Sample random context features."""
        # Volatility z-score: -2 to 2
        vol = np.random.uniform(-1.5, 1.5)
        # Term structure z-score
        term = np.random.uniform(-1.0, 1.0)
        # Seasonality: -1 to 1
        season = np.random.uniform(-1.0, 1.0)
        return np.array([vol, term, season])
    
    def get_reward(self, arm, context):
        """Linear reward: r = context^T theta_arm + noise."""
        mean_reward = context @ self.true_theta[arm]
        noise = np.random.normal(0, self.noise_std)
        return mean_reward + noise
    
    def get_best_arm(self, context):
        """Oracle: which arm is actually best for this context?"""
        expected_rewards = [context @ self.true_theta[a] for a in range(3)]
        return np.argmax(expected_rewards)

# Test environment
env = LinearCommodityEnvironment()
context = env.get_context()
print(f"Context: {context}")
print(f"Expected rewards: {[context @ env.true_theta[a] for a in range(3)]}")
print(f"Best arm: {env.arms[env.get_best_arm(context)]}")

## Cell 4: Run LinUCB and Watch It Learn

See how LinUCB discovers the linear relationship between features and rewards.

In [ ]:
# Initialize
env = LinearCommodityEnvironment()
bandit = LinUCB(n_arms=3, context_dim=3, alpha=1.0)

# Run simulation
T = 1000
rewards = []
regrets = []
arm_choices = []

for t in range(T):
    # Get context
    context = env.get_context()
    
    # Choose arm
    arm = bandit.choose_arm(context)
    arm_choices.append(arm)
    
    # Get reward
    reward = env.get_reward(arm, context)
    rewards.append(reward)
    
    # Compute regret (vs. oracle)
    best_arm = env.get_best_arm(context)
    best_reward = context @ env.true_theta[best_arm]
    actual_reward = context @ env.true_theta[arm]
    regret = best_reward - actual_reward
    regrets.append(regret)
    
    # Update bandit
    bandit.update(arm, context, reward)

print(f"Total reward: {sum(rewards):.2f}")
print(f"Cumulative regret: {sum(regrets):.2f}")
print(f"Average regret per round: {np.mean(regrets):.4f}")
print(f"\nArm selection counts: {[arm_choices.count(i) for i in range(3)]}")

## Cell 5: Visualize Learned Weights

Compare the weights LinUCB learned to the true weights.

In [ ]:
# Extract learned weights
learned_weights = np.array([bandit.get_weights(a) for a in range(3)])
true_weights = env.true_theta

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
feature_names = ['Volatility', 'Term Structure', 'Seasonality']

for arm in range(3):
    x = np.arange(3)
    width = 0.35
    
    axes[arm].bar(x - width/2, true_weights[arm], width, label='True', alpha=0.8)
    axes[arm].bar(x + width/2, learned_weights[arm], width, label='Learned', alpha=0.8)
    axes[arm].set_xlabel('Feature')
    axes[arm].set_ylabel('Weight')
    axes[arm].set_title(f'{env.arms[arm]}')
    axes[arm].set_xticks(x)
    axes[arm].set_xticklabels(feature_names, rotation=45, ha='right')
    axes[arm].legend()
    axes[arm].grid(True, alpha=0.3, axis='y')
    axes[arm].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

# Compute weight recovery error
weight_error = np.abs(learned_weights - true_weights).mean()
print(f"\nAverage weight error: {weight_error:.4f}")
print(f"LinUCB successfully recovered the linear structure!")

## Cell 6: Modify This — Explore Alpha Parameter

See how the exploration parameter α affects learning.

In [ ]:
# Test different alpha values
alphas = [0.1, 0.5, 1.0, 2.0, 5.0]
results = {}

for alpha in alphas:
    env = LinearCommodityEnvironment()
    bandit = LinUCB(n_arms=3, context_dim=3, alpha=alpha)
    
    regrets_alpha = []
    for t in range(500):
        context = env.get_context()
        arm = bandit.choose_arm(context)
        reward = env.get_reward(arm, context)
        bandit.update(arm, context, reward)
        
        best_arm = env.get_best_arm(context)
        best_reward = context @ env.true_theta[best_arm]
        actual_reward = context @ env.true_theta[arm]
        regrets_alpha.append(best_reward - actual_reward)
    
    results[alpha] = np.cumsum(regrets_alpha)

# Plot regret curves
plt.figure(figsize=(10, 6))
for alpha, cumregret in results.items():
    plt.plot(cumregret, label=f'α={alpha}', linewidth=2)

plt.xlabel('Round')
plt.ylabel('Cumulative Regret')
plt.title('Effect of Exploration Parameter α on LinUCB Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations:")
print("- α too small (0.1): Fast initial learning but suboptimal long-term (insufficient exploration)")
print("- α too large (5.0): Slow learning due to excessive exploration")
print("- α=1.0 is a good default balance")
print("\nModify This: Change T, add features, or try different true_theta values!")

## Cell 7: Performance Analysis

Analyze LinUCB's decision quality over time.

In [ ]:
# Run again with tracking
env = LinearCommodityEnvironment()
bandit = LinUCB(n_arms=3, context_dim=3, alpha=1.0)

T = 1000
accuracy_over_time = []
window = 50

correct_choices = []

for t in range(T):
    context = env.get_context()
    arm = bandit.choose_arm(context)
    best_arm = env.get_best_arm(context)
    
    correct = int(arm == best_arm)
    correct_choices.append(correct)
    
    # Rolling accuracy
    if t >= window:
        accuracy_over_time.append(np.mean(correct_choices[-window:]))
    
    reward = env.get_reward(arm, context)
    bandit.update(arm, context, reward)

# Plot learning curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy over time
axes[0].plot(range(window, T), accuracy_over_time, linewidth=2)
axes[0].axhline(y=1/3, color='r', linestyle='--', label='Random baseline')
axes[0].set_xlabel('Round')
axes[0].set_ylabel(f'Accuracy (rolling {window} rounds)')
axes[0].set_title('LinUCB Learns to Choose Optimal Arm')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Regret growth rate
regrets = []
for t in range(T):
    context = env.get_context()
    best_arm = env.get_best_arm(context)
    # Use current policy
    arm = np.argmax([context @ bandit.get_weights(a) for a in range(3)])
    regret = context @ (env.true_theta[best_arm] - env.true_theta[arm])
    regrets.append(max(0, regret))

axes[1].plot(np.cumsum(regrets), linewidth=2)
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Cumulative Regret')
axes[1].set_title('Regret Growth Slows as Learning Improves')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_accuracy = np.mean(correct_choices[-100:])
print(f"\nFinal accuracy (last 100 rounds): {final_accuracy:.1%}")
print(f"LinUCB learned to choose the optimal arm in most contexts!")

## Summary and Key Takeaways

**What we learned:**

1. **LinUCB combines prediction and exploration** — Ridge regression for expected reward + confidence bounds for uncertainty

2. **It recovers the linear structure** — Learned weights closely match true weights after sufficient data

3. **Alpha controls exploration** — Too small = exploitation-heavy, too large = exploration-heavy; α=1.0 is a good default

4. **Regret is sublinear** — Cumulative regret grows slower over time as the model improves

5. **Accuracy improves to 80-90%+** — After learning, LinUCB chooses the optimal arm in most contexts

**When to use LinUCB:**
- Rewards have approximately linear relationship with features
- Context dimension is small (d < 20)
- You need interpretable models (can inspect learned weights)
- Computational efficiency matters (O(d³) per decision)

**Next steps:**
- Notebook 03: Apply LinUCB to real commodity data with market regime features
- Try: Kernel LinUCB for nonlinear relationships, Thompson Sampling for Bayesian alternative

**Modify This:**
- Increase context_dim to 5 and add more features
- Make the true relationship nonlinear (e.g., add quadratic terms) and see LinUCB struggle
- Implement a random baseline and compare performance
- Add time-varying noise or non-stationary rewards

In [ ]:
key_takeaways(["LinUCB combines prediction and exploration", "It recovers the linear structure", "Alpha controls exploration", "Regret is sublinear", "Accuracy improves to 80-90%+"])